# Colab setup: Search_AI_project (OTTO recsys)

Run this once at the start of every Colab session. It:
1. Mounts Drive and clones/syncs the repo there (so code + git history persist across sessions)
2. Installs the same Python environment as local (via `uv`)
3. Sets up Kaggle credentials and downloads the **full** OTTO dataset (~11GB) to Drive, so it's
   downloaded once and reused across every future session instead of re-downloading each time -
   this project will likely span many sessions (Two-Tower, DCN-V2, MMoE, ensembling), so paying
   the download cost once is worth Drive's slower-than-local I/O. Skips the download automatically
   if the file's already there.

Before running: make sure you have a GPU runtime (Runtime -> Change runtime type -> GPU).

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from pathlib import Path

REPO_URL = "https://github.com/shaliqsv/Search_AI_project.git"
REPO_ROOT = Path('/content/drive/MyDrive/search_ai_project')

if not (REPO_ROOT / "pyproject.toml").exists():
    print(f"Repo not found at {REPO_ROOT} - cloning ...")
    !git clone "{REPO_URL}" "{REPO_ROOT}"
else:
    # see financebench_project's setup cell for why: self-heals a detached HEAD,
    # and --no-rebase --no-edit self-heals a diverged local/origin history
    # without hanging on an editor prompt inside a notebook `!` cell
    print(f"Repo already present at {REPO_ROOT} - syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

print(f"REPO_ROOT: {REPO_ROOT}")

Repo already present at /content/drive/MyDrive/search_ai_project - syncing to latest main ...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.35 KiB | 26.00 KiB/s, done.
From https://github.com/shaliqsv/Search_AI_project
   d8246ba..00ef79e  main       -> origin/main
Already on 'main'
Your branch is behind 'origin/main' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/shaliqsv/Search_AI_project
 * branch            main       -> FETCH_HEAD
Updating d8246ba..00ef79e
Fast-forward
 notebooks/colab/00_colab_setup.ipynb | 44 ++++--------------------------------
 1 file changed, 4 insertions(+), 40 deletions(-)
REPO_ROOT: /content/drive/MyDrive/search_ai_project


## Install the environment

Colab already ships numpy/pandas/torch. We only need to add what it doesn't have. Installing
straight into Colab's own environment rather than a `uv` venv on the Drive mount deliberately -
a venv is thousands of small files, and Drive's FUSE mount is slow and occasionally flaky for
that pattern. The whole Colab runtime is already disposable/isolated, so skipping the extra
venv layer is the standard approach here, not a shortcut.

In [7]:
!pip install -q polars duckdb lightgbm kaggle

import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
print("environment ready")

environment ready


## Download the full OTTO dataset

Downloaded once to Drive (persists across sessions - see the note at the top). CC BY 4.0, sourced
from the dataset endpoint not the competition - see `_doc/decisions/D1-otto-license.md` for why
that distinction matters.

In [9]:
DATA_DIR = Path('/content/drive/MyDrive/otto_data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

train_file = DATA_DIR / "otto-recsys-train.jsonl"
if train_file.exists():
    size_gb = train_file.stat().st_size / 1e9
    print(f"Already downloaded: {train_file} ({size_gb:.1f} GB) - skipping")
else:
    !kaggle datasets download -d otto/recsys-dataset -p "{DATA_DIR}" --unzip

!ls -lh "{DATA_DIR}"

Dataset URL: https://www.kaggle.com/datasets/otto/recsys-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 1.98G/1.98G [01:50<00:00, 19.2MB/s]

total 12G
-rw------- 1 root root 716M Oct  2 08:14 otto-recsys-test.jsonl
-rw------- 1 root root  11G Oct  2 08:15 otto-recsys-train.jsonl


In [10]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: NVIDIA A100-SXM4-40GB


## Next steps

With the full dataset at `/content/drive/MyDrive/otto_data/otto-recsys-train.jsonl` (persisted on
Drive), the Phase 1 notebook's sampling step can either be skipped (use the full file directly, now
that RAM/disk aren't the constraint they were locally) or re-run at a much higher keep-percentage.
From there, Phase 1-7 notebooks port over unchanged (same `src/ranking` code, same logic) - the only
difference is scale and that `ROOT` now resolves to this Drive-mounted repo instead of the local
Mac path.

Two-Tower training (PyTorch, GPU) is the next real piece of new work once the data's in place.